In [1]:
import pandas as pd
import ollama
from sqlalchemy import text

from pipeline_utils import (
    engine,
    get_customer_history,
    find_similar_ticket,
    analyze_ticket,
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
def generate_recommendation(ticket_id):

    # Pull current ticket's core info
    ticket_query = text("""
        SELECT `Ticket ID`, `Customer ID`, `Product Purchased`, `Ticket Description`, `Ticket Priority`
        FROM tickets
        WHERE `Ticket ID` = :ticket_id
    """)
    ticket_row = pd.read_sql(ticket_query, con=engine, params={"ticket_id": ticket_id}).iloc[0]

    customer_id = ticket_row["Customer ID"]
    description = ticket_row["Ticket Description"]

    # Gather context from Stages 2-4
    analysis = analyze_ticket(ticket_id)
    history = get_customer_history(customer_id, ticket_id)
    similar = find_similar_ticket(description, ticket_id, customer_id=customer_id)

    # Build the similar ticket section conditionally
    if similar["found"]:
        similar_section = f"""Found: Yes
Similarity: {similar['similarity']}
Previous issue: {similar['previous_description']}
Search scope: {similar['search_scope']}"""
    else:
        similar_section = "Found: No similar previous ticket"

    prompt = f"""You are assisting a customer support agent. Based ONLY on the context below, provide:
1. A brief summary of the customer's issue
2. A recommended action for the agent to take
3. A short reason explaining why you recommend that action

CURRENT TICKET
---------------
Product: {ticket_row['Product Purchased']}
Description: {description}
Priority: {analysis['priority']}
Urgency: {analysis['urgency']}

CUSTOMER HISTORY
----------------
Previous tickets: {history['previous_ticket_count']}
Unresolved tickets: {history['unresolved_ticket_count']}
Previous products: {history['previous_products']}
Previous satisfaction ratings: {history['previous_satisfaction']}
Previously purchased this product: {history['previously_purchased_current_product']}

SIMILAR PREVIOUS TICKET
------------------------
{similar_section}

STRICT RULES:
- Only reference facts that appear explicitly above. Do not mention policies, refund rules, warranty terms, wholesale/retail distinctions, or resolutions that are not written above.
- If no similar previous ticket was found, do not invent one or reference a "previous resolution."
- If the context does not clearly support a specific action, recommend that the agent gather more information from the customer rather than guessing a resolution.
- Keep the response short: 1-2 sentences per section."""

    response = ollama.chat(
        model="llama3.2",
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.2}
    )

    return {
        "ticket_id": ticket_id,
        "recommendation_text": response["message"]["content"].strip(),
        "context_used": {
            "analysis": analysis,
            "history": history,
            "similar_ticket": similar
        }
    }

In [5]:
result = generate_recommendation(4491)
print(result["recommendation_text"])

Here are the recommended actions and reasons:

1. Brief summary of the customer's issue:
The customer is experiencing an issue with their GoPro Action Camera, where the price does not reflect the price they paid when purchasing the product.

2. Recommended action for the agent to take:
Gather more information from the customer about the specific settings and configurations they have tried, and the exact issue they are experiencing.

Reason: The customer has already tried troubleshooting steps mentioned in the user manual, but the issue persists. Gathering more information will help the agent understand the specific problem and provide a more accurate resolution.


In [6]:
result = generate_recommendation(7652)
print(result["recommendation_text"])

Here are the recommended actions:

1. Brief summary of the customer's issue: The customer is experiencing a critical issue with their Canon EOS camera after a recent software update, and the problem started occurring after the update.

2. Recommended action for the agent to take: Gather more information from the customer about the issue, such as the specific error messages or symptoms they are experiencing, to better understand the problem and provide a more effective resolution.

3. Short reason explaining why I recommend that action: The customer's issue is described as "Critical" and "Urgent", suggesting that a prompt and accurate diagnosis is essential to resolve the problem quickly. Gathering more information will help the agent to provide a more accurate and effective resolution.


In [7]:
result = generate_recommendation(1153)
print(result["recommendation_text"])

Here are the recommended actions and reasons:

1. Brief summary of the customer's issue:
The customer's LG OLED TV is experiencing intermittent flickering issues, making it unusable.

2. Recommended action for the agent to take:
Schedule a repair or replacement for the TV.

Reason: The customer has already tried troubleshooting with multiple tools, indicating a hardware problem that requires professional attention.


In [12]:
def generate_customer_response(ticket_id):

    ticket_query = text("""
        SELECT `Ticket ID`, `Customer ID`, `Customer Name`, `Product Purchased`, `Ticket Description`
        FROM tickets
        WHERE `Ticket ID` = :ticket_id
    """)
    ticket_row = pd.read_sql(ticket_query, con=engine, params={"ticket_id": ticket_id}).iloc[0]

    customer_id = ticket_row["Customer ID"]
    description = ticket_row["Ticket Description"]

    history = get_customer_history(customer_id, ticket_id)
    similar = find_similar_ticket(description, ticket_id, customer_id=customer_id)

    if similar["found"]:
        if similar["search_scope"] == "same_customer":
            similar_section = f"""This customer has a similar previous issue on file:
    Previous issue: {similar['previous_description']}"""
        else:
            similar_section = f"""A similar issue has been reported by another customer (not this customer's own history):
    Previous issue: {similar['previous_description']}"""
    else:
        similar_section = "No similar previous ticket was found."

    prompt = f"""Write a short, professional customer support reply to the customer below.

CUSTOMER'S ISSUE
-----------------
Product: {ticket_row['Product Purchased']}
Description: {description}

CUSTOMER CONTEXT
------------------
Previously purchased this product before: {history['previously_purchased_current_product']}
Unresolved tickets on file: {history['unresolved_ticket_count']}

RELATED HISTORY
----------------
{similar_section}

STRICT RULES:
- Do not promise a specific resolution (refund, replacement, repair) unless one is explicitly stated above.
- Do not invent policy details, timelines, or outcomes not present above.
- Acknowledge the issue, reassure the customer it's being looked into, and if related history exists, you may mention that a similar case is being reviewed.
- Keep it to 3-4 sentences, professional and empathetic tone, no signature/closing needed.
- Do not use the customer's name if uncertain of it; a generic greeting is fine.
- Only say "your previous case" or refer to the customer's own history if the related history above is explicitly from this same customer. 
If the similar issue is from another customer, refer to it only as "a known issue" or "similar cases we've seen," never as belonging to this customer."""

    response = ollama.chat(
        model="llama3.2",
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.3}
    )

    return {
        "ticket_id": ticket_id,
        "suggested_response": response["message"]["content"].strip()
    }

In [13]:
result = generate_customer_response(4491)
print(result["suggested_response"])

Dear Valued Customer,

Thank you for reaching out to us regarding the issue with your GoPro Action Camera. We apologize for the inconvenience and appreciate you bringing this to our attention. We're looking into this matter and will provide an update as soon as possible. Our team is reviewing a known issue with similar cases we've seen, and we'll do our best to assist you in resolving the problem.

Best regards.


In [14]:
result = generate_customer_response(7652)
print(result["suggested_response"])

Dear Customer,

Thank you for reaching out to us regarding the issue with your Canon EOS. We apologize for the inconvenience caused by the recent software update. Our team is currently investigating this issue and will look into possible solutions. We appreciate your patience and will keep you updated on any progress.

Best regards.


In [15]:
result = generate_customer_response(1153)
print(result["suggested_response"])

Dear valued customer,

We apologize for the inconvenience you're experiencing with your LG OLED screen flickering and not functioning properly. We've taken note of your issue and are currently investigating the matter to determine the cause and potential solution. Our team is reviewing a known issue with similar symptoms on multiple devices of the same model, and we'll provide an update as soon as possible. We appreciate your patience and understanding as we work to resolve this issue.

Best regards,
Customer Support Team
